# Multi-Language Code Generator

Port Python code to **5 target languages** — C++, Rust, C, Go, and Java — using frontier LLMs.

For each language we:
1. Check system readiness (toolchain detection)
2. Use an LLM to generate compile & run commands
3. Port Python → target language
4. Generate unit tests for the ported code
5. Compile and run both the ported code and its tests

Built on top of the day3→day5 progression, now unified in a single Gradio UI with tabbed Code / Tests views.

In [ ]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

for name, key, prefix_len in [
    ("OpenAI", openai_api_key, 8),
    ("Anthropic", anthropic_api_key, 7),
    ("Google", google_api_key, 2),
    ("Groq", groq_api_key, 4),
    ("OpenRouter", openrouter_api_key, 6),
]:
    if key:
        print(f"{name} API Key exists and begins {key[:prefix_len]}")
    else:
        print(f"{name} API Key not set")

In [ ]:
openai_client = OpenAI()

anthropic = OpenAI(api_key=anthropic_api_key, base_url="https://api.anthropic.com/v1/")
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
openrouter = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")

models = [
    "gpt-5",
    "gemini-2.5-pro",
    "claude-opus-4-6",
    "qwen/qwen3-coder-next",
    "deepseek/deepseek-chat-v3-0324",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
]

clients = {
    "gpt-5": openai_client,
    "claude-opus-4-6": anthropic,
    "gemini-2.5-pro": gemini,
    "openai/gpt-oss-120b": openrouter,
    "qwen/qwen3-coder-next": openrouter,
    "deepseek/deepseek-chat-v3-0324": openrouter,
    "openai/gpt-oss-20b": openrouter,
}

## System Readiness Checks

Detect what toolchains are available on this machine for all 5 target languages.

In [ ]:
from system_info import retrieve_system_info, rust_toolchain_info, c_toolchain_info, go_toolchain_info, java_toolchain_info

system_info = retrieve_system_info()
system_info

In [ ]:
cpp_info = system_info["toolchain"]
print("C++ toolchain:")
cpp_info

In [ ]:
rust_info = rust_toolchain_info()
print(f"Rust installed: {rust_info['installed']}")
rust_info

In [ ]:
c_info = c_toolchain_info()
print(f"C compiler installed: {c_info['installed']}")
c_info

In [ ]:
go_info = go_toolchain_info()
print(f"Go installed: {go_info['installed']}")
go_info

In [ ]:
java_info = java_toolchain_info()
print(f"Java installed: {java_info['installed']}")
java_info

## Generate Compile & Run Commands

For each target language, ask the LLM to tell us:
- Whether we need to install anything
- The exact `compile_command` and `run_command` (as Python lists for `subprocess.run`)
- The exact `test_compile_command` and `test_run_command` for running unit tests

Run each cell below, read the LLM response, and fill in the command variables in the section that follows.

In [ ]:
def build_command_prompt(language, source_file, output_binary, toolchain_info,
                         test_source_file=None, test_binary=None):
    """Build a prompt asking the LLM for compile/run commands for a given language."""
    test_section = ""
    if test_source_file:
        test_section = f"""
I also need commands to compile and run a **unit test file** called `{test_source_file}` that tests the code in `{source_file}`.
The test binary should be called `{test_binary}`.
Please provide `test_compile_command` and `test_run_command` as Python lists too.
"""
    return f"""
Here is a report of the system information for my computer.
I want to compile a single {language} file called `{source_file}` and then execute it for the fastest possible runtime performance.

Please reply with whether I need to install any {language} toolchain to do this. If so, provide the simplest step-by-step instructions.

If I'm already set up, I'd like to run something like this in Python:
```python
compile_command = [...]  # for fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = [...]
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
```

Please tell me exactly what I should use for `compile_command` and `run_command` as Python lists.
{test_section}
System information:
{system_info}

{language} toolchain information:
{toolchain_info}
"""

### C++ — compile & run commands

In [ ]:
cpp_prompt = build_command_prompt(
    language="C++",
    source_file="main.cpp",
    output_binary="main_cpp",
    toolchain_info=cpp_info,
    test_source_file="test_main.cpp",
    test_binary="test_main_cpp",
)
response = openai_client.chat.completions.create(
    model="gpt-5", messages=[{"role": "user", "content": cpp_prompt}]
)
display(Markdown(response.choices[0].message.content))

### Rust — compile & run commands

In [ ]:
rust_prompt = build_command_prompt(
    language="Rust",
    source_file="main.rs",
    output_binary="main_rs",
    toolchain_info=rust_info,
    test_source_file="test_main.rs",
    test_binary="test_main_rs",
)
response = openai_client.chat.completions.create(
    model="gpt-5", messages=[{"role": "user", "content": rust_prompt}]
)
display(Markdown(response.choices[0].message.content))

### C — compile & run commands

In [ ]:
c_prompt = build_command_prompt(
    language="C",
    source_file="main.c",
    output_binary="main_c",
    toolchain_info=c_info,
    test_source_file="test_main.c",
    test_binary="test_main_c",
)
response = openai_client.chat.completions.create(
    model="gpt-5", messages=[{"role": "user", "content": c_prompt}]
)
display(Markdown(response.choices[0].message.content))

### Go — compile & run commands

In [ ]:
go_prompt = build_command_prompt(
    language="Go",
    source_file="main.go",
    output_binary="main_go",
    toolchain_info=go_info,
    test_source_file="main_test.go",
    test_binary="(go test)",
)
response = openai_client.chat.completions.create(
    model="gpt-5", messages=[{"role": "user", "content": go_prompt}]
)
display(Markdown(response.choices[0].message.content))

### Java — compile & run commands

In [ ]:
java_prompt = build_command_prompt(
    language="Java",
    source_file="Main.java",
    output_binary="Main (class)",
    toolchain_info=java_info,
    test_source_file="MainTest.java",
    test_binary="MainTest (class)",
)
response = openai_client.chat.completions.create(
    model="gpt-5", messages=[{"role": "user", "content": java_prompt}]
)
display(Markdown(response.choices[0].message.content))

## Set Compile & Run Commands

Based on the LLM responses above, fill in the commands below for each language.
The example values are sensible defaults for macOS with Apple Clang / Homebrew toolchains — **adjust to match what the LLM told you**.

In [ ]:
# ── C++ commands ──
cpp_compile = ["clang++", "-std=c++20", "-O3", "-ffast-math", "-flto", "-DNDEBUG", "main.cpp", "-o", "main_cpp"]
cpp_run = ["./main_cpp"]
cpp_test_compile = ["clang++", "-std=c++20", "-O3", "-DNDEBUG", "test_main.cpp", "-o", "test_main_cpp"]
cpp_test_run = ["./test_main_cpp"]

print("C++ compile:", " ".join(cpp_compile))
print("C++ test compile:", " ".join(cpp_test_compile))

In [ ]:
# ── Rust commands ──
rustc = rust_info["rustc"]["path"] or "rustc"

rust_compile = [rustc, "main.rs", "-C", "opt-level=3", "-C", "lto=fat", "-C", "codegen-units=1", "-C", "target-cpu=native", "-C", "panic=abort", "-o", "main_rs"]
rust_run = ["./main_rs"]
rust_test_compile = [rustc, "test_main.rs", "--test", "-C", "opt-level=2", "-o", "test_main_rs"]
rust_test_run = ["./test_main_rs"]

print("Rust compile:", " ".join(rust_compile))
print("Rust test compile:", " ".join(rust_test_compile))

In [ ]:
# ── C commands ──
cc = c_info["preferred_compiler"] or "gcc"
cc_path = c_info["compilers"].get(cc, {}).get("path", cc) if c_info["compilers"] else cc

c_compile = [cc_path, "-std=c17", "-O3", "-ffast-math", "-flto", "-DNDEBUG", "main.c", "-o", "main_c", "-lm"]
c_run = ["./main_c"]
c_test_compile = [cc_path, "-std=c17", "-O3", "-DNDEBUG", "test_main.c", "-o", "test_main_c", "-lm"]
c_test_run = ["./test_main_c"]

print("C compile:", " ".join(c_compile))
print("C test compile:", " ".join(c_test_compile))

In [ ]:
# ── Go commands ──
go_bin = go_info["go"]["path"] or "go"

go_compile = [go_bin, "build", "-o", "main_go", "main.go"]
go_run = ["./main_go"]
go_test_compile = []  # Go tests don't need a separate compile step
go_test_run = [go_bin, "test", "-v", "-run", ".", "."]

print("Go compile:", " ".join(go_compile))
print("Go test run:", " ".join(go_test_run))

In [ ]:
# ── Java commands ──
javac_bin = java_info["javac"]["path"] or "javac"
java_bin = java_info["java"]["path"] or "java"

java_compile = [javac_bin, "Main.java"]
java_run = [java_bin, "Main"]
java_test_compile = [javac_bin, "MainTest.java"]
java_test_run = [java_bin, "MainTest"]

print("Java compile:", " ".join(java_compile))
print("Java test compile:", " ".join(java_test_compile))

## Language Registry & Core Functions

A single dict maps each language to its file names, compile/run commands, and code-fence markers used by LLMs.

In [ ]:
LANGUAGES = {
    "C++": {
        "extension": "cpp",
        "source_file": "main.cpp",
        "test_file": "test_main.cpp",
        "compile": cpp_compile,
        "run": cpp_run,
        "test_compile": cpp_test_compile,
        "test_run": cpp_test_run,
        "fence_tags": ["```cpp", "```c++"],
        "gradio_lang": "cpp",
    },
    "Rust": {
        "extension": "rs",
        "source_file": "main.rs",
        "test_file": "test_main.rs",
        "compile": rust_compile,
        "run": rust_run,
        "test_compile": rust_test_compile,
        "test_run": rust_test_run,
        "fence_tags": ["```rust"],
        "gradio_lang": "rust",
    },
    "C": {
        "extension": "c",
        "source_file": "main.c",
        "test_file": "test_main.c",
        "compile": c_compile,
        "run": c_run,
        "test_compile": c_test_compile,
        "test_run": c_test_run,
        "fence_tags": ["```c"],
        "gradio_lang": "c",
    },
    "Go": {
        "extension": "go",
        "source_file": "main.go",
        "test_file": "main_test.go",
        "compile": go_compile,
        "run": go_run,
        "test_compile": go_test_compile,
        "test_run": go_test_run,
        "fence_tags": ["```go"],
        "gradio_lang": "go",
    },
    "Java": {
        "extension": "java",
        "source_file": "Main.java",
        "test_file": "MainTest.java",
        "compile": java_compile,
        "run": java_run,
        "test_compile": java_test_compile,
        "test_run": java_test_run,
        "fence_tags": ["```java"],
        "gradio_lang": "java",
    },
}

language_names = list(LANGUAGES.keys())
print("Registered languages:", language_names)

In [ ]:
def strip_fences(code, lang_config):
    """Remove markdown code fences the LLM wraps around generated code."""
    for tag in lang_config["fence_tags"]:
        code = code.replace(tag, "")
    code = code.replace("```", "")
    return code.strip()


def make_port_prompt(language, lang_config, python_code):
    return f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.

Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called {lang_config['source_file']} and then compiled and executed.
The compilation command is: {lang_config['compile']}
Respond only with {language} code.

Python code to port:

```python
{python_code}
```
"""


def make_test_prompt(language, lang_config, ported_code, python_code):
    return f"""
Your task is to write unit tests for the following {language} code that was ported from Python.
The tests should verify that the ported code produces correct results.

Write the tests in a **separate file** called `{lang_config['test_file']}`.
The test file should be self-contained — include any necessary imports and a main function.
Do NOT use external test frameworks that need installation; use only what ships with the language:
- C++: use assert() from <cassert> with a main() that runs all tests
- Rust: use #[cfg(test)] mod tests with #[test] functions and assert! / assert_eq! macros
- C: use assert() from <assert.h> with a main() that runs all tests
- Go: use the standard "testing" package with func TestXxx(t *testing.T)
- Java: use simple assert statements or if/throw in a main() method (no JUnit)

The test file will be compiled with: {lang_config.get('test_compile', 'N/A')}
And run with: {lang_config.get('test_run', 'N/A')}

Respond only with {language} code for the test file.

Original Python code:
```python
{python_code}
```

Ported {language} code:
```
{ported_code}
```
"""

In [ ]:
def run_python(code):
    """Execute Python code and capture stdout."""
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    return output


def write_file(filename, content):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)


def port_code(model, language, python_code):
    """Ask the LLM to port Python code to the target language. Returns the generated code."""
    lang_config = LANGUAGES[language]
    client = clients[model]
    reasoning_effort = "high" if "gpt" in model else None

    prompt = make_port_prompt(language, lang_config, python_code)
    messages = [
        {"role": "system", "content": f"You convert Python code into high performance {language} code. Respond only with {language} code."},
        {"role": "user", "content": prompt},
    ]

    response = client.chat.completions.create(model=model, messages=messages, reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    return strip_fences(reply, lang_config)


def generate_tests(model, language, ported_code, python_code):
    """Ask the LLM to generate unit tests for the ported code. Returns the test code."""
    lang_config = LANGUAGES[language]
    client = clients[model]
    reasoning_effort = "high" if "gpt" in model else None

    prompt = make_test_prompt(language, lang_config, ported_code, python_code)
    messages = [
        {"role": "system", "content": f"You write unit tests for {language} code. Respond only with {language} test code."},
        {"role": "user", "content": prompt},
    ]

    response = client.chat.completions.create(model=model, messages=messages, reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    return strip_fences(reply, lang_config)


def compile_and_run(language, code):
    """Write source, compile, and run. Returns stdout or error."""
    lang_config = LANGUAGES[language]
    write_file(lang_config["source_file"], code)

    try:
        if lang_config["compile"]:
            subprocess.run(lang_config["compile"], check=True, text=True, capture_output=True)
        result = subprocess.run(lang_config["run"], check=True, text=True, capture_output=True)
        return result.stdout
    except subprocess.CalledProcessError as e:
        return f"Error:\n{e.stderr}"


def compile_and_run_tests(language, test_code):
    """Write test source, compile, and run tests. Returns stdout or error."""
    lang_config = LANGUAGES[language]
    write_file(lang_config["test_file"], test_code)

    try:
        if lang_config["test_compile"]:
            subprocess.run(lang_config["test_compile"], check=True, text=True, capture_output=True)
        result = subprocess.run(lang_config["test_run"], check=True, text=True, capture_output=True)
        return result.stdout + (result.stderr or "")
    except subprocess.CalledProcessError as e:
        return f"Test Error:\n{e.stdout}\n{e.stderr}"

## Sample Python Code

In [ ]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

## Gradio UI

Select a target language from the dropdown, port the code, generate unit tests, and run everything — all from one interface.

In [ ]:
from styles import CSS


def run_python(code):
    """Execute Python code and capture stdout."""
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    return output


def write_file(filename, content):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)


NATIVE_OPENAI_MODELS = {"gpt-5", "gpt-5-nano", "gpt-5-mini"}


def _call_llm(client, model, messages):
    """Call the LLM with the right kwargs per model. Returns the reply text."""
    kwargs = dict(model=model, messages=messages)
    if model in NATIVE_OPENAI_MODELS:
        kwargs["reasoning_effort"] = "high"
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content


def port_code(model, language, python_code):
    """Ask the LLM to port Python code to the target language. Returns the generated code."""
    try:
        lang_config = LANGUAGES[language]
        client = clients[model]

        prompt = make_port_prompt(language, lang_config, python_code)
        messages = [
            {"role": "system", "content": f"You convert Python code into high performance {language} code. Respond only with {language} code."},
            {"role": "user", "content": prompt},
        ]

        reply = _call_llm(client, model, messages)
        return strip_fences(reply, lang_config)
    except Exception as e:
        return f"Error porting code: {e}"


def generate_tests(model, language, ported_code, python_code):
    """Ask the LLM to generate unit tests for the ported code. Returns the test code."""
    try:
        lang_config = LANGUAGES[language]
        client = clients[model]

        prompt = make_test_prompt(language, lang_config, ported_code, python_code)
        messages = [
            {"role": "system", "content": f"You write unit tests for {language} code. Respond only with {language} test code."},
            {"role": "user", "content": prompt},
        ]

        reply = _call_llm(client, model, messages)
        return strip_fences(reply, lang_config)
    except Exception as e:
        return f"Error generating tests: {e}"


def compile_and_run(language, code):
    """Write source, compile, and run. Returns stdout or error."""
    lang_config = LANGUAGES[language]
    write_file(lang_config["source_file"], code)

    try:
        if lang_config["compile"]:
            subprocess.run(lang_config["compile"], check=True, text=True, capture_output=True)
        result = subprocess.run(lang_config["run"], check=True, text=True, capture_output=True)
        return result.stdout
    except subprocess.CalledProcessError as e:
        return f"Error:\n{e.stderr}"


def compile_and_run_tests(language, test_code):
    """Write test source, compile, and run tests. Returns stdout or error."""
    lang_config = LANGUAGES[language]
    write_file(lang_config["test_file"], test_code)

    try:
        if lang_config["test_compile"]:
            subprocess.run(lang_config["test_compile"], check=True, text=True, capture_output=True)
        result = subprocess.run(lang_config["test_run"], check=True, text=True, capture_output=True)
        return result.stdout + (result.stderr or "")
    except subprocess.CalledProcessError as e:
        return f"Test Error:\n{e.stdout}\n{e.stderr}"


with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Multi-Language Code Generator") as ui:
    gr.Markdown("# Multi-Language Code Generator\nPort Python → C++ / Rust / C / Go / Java, with auto-generated unit tests")

    with gr.Row(equal_height=True):
        model = gr.Dropdown(models, value=models[0], label="LLM Model", scale=3)
        language = gr.Dropdown(language_names, value=language_names[0], label="Target Language", scale=2)

    with gr.Tabs():
        with gr.TabItem("Code"):
            with gr.Row(equal_height=True):
                with gr.Column(scale=6):
                    python_code = gr.Code(
                        label="Python (original)",
                        value=python_hard,
                        language="python",
                        lines=26,
                    )
                with gr.Column(scale=6):
                    ported_code = gr.Code(
                        label="Ported code (generated)",
                        value="",
                        language="cpp",
                        lines=26,
                    )

            with gr.Row(elem_classes=["controls"]):
                python_run_btn = gr.Button("Run Python", elem_classes=["run-btn", "py"])
                port_btn = gr.Button("Port Code", elem_classes=["convert-btn"])
                lang_run_btn = gr.Button("Run Ported Code", elem_classes=["run-btn", "cpp"])

            with gr.Row(equal_height=True):
                with gr.Column(scale=6):
                    python_out = gr.TextArea(label="Python result", lines=6, elem_classes=["py-out"])
                with gr.Column(scale=6):
                    lang_out = gr.TextArea(label="Ported code result", lines=6, elem_classes=["lang-out"])

        with gr.TabItem("Tests"):
            with gr.Row(equal_height=True):
                with gr.Column(scale=6):
                    test_code = gr.Code(
                        label="Unit tests (generated)",
                        value="",
                        language="cpp",
                        lines=26,
                    )
                with gr.Column(scale=6):
                    test_out = gr.TextArea(label="Test results", lines=26, elem_classes=["test-out"])

            with gr.Row(elem_classes=["controls"]):
                gen_tests_btn = gr.Button("Generate Tests", elem_classes=["convert-btn"])
                run_tests_btn = gr.Button("Run Tests", elem_classes=["test-btn"])

    port_btn.click(fn=port_code, inputs=[model, language, python_code], outputs=[ported_code])
    gen_tests_btn.click(fn=generate_tests, inputs=[model, language, ported_code, python_code], outputs=[test_code])
    python_run_btn.click(fn=run_python, inputs=[python_code], outputs=[python_out])
    lang_run_btn.click(fn=compile_and_run, inputs=[language, ported_code], outputs=[lang_out])
    run_tests_btn.click(fn=compile_and_run_tests, inputs=[language, test_code], outputs=[test_out])

ui.launch(inbrowser=True)

## Results

Use this space to record your performance results across languages and models.

### C++

| Model | Execution Time | Speedup vs Python |
|-------|---------------|-------------------|
| gpt-5 | | |
| gemini-2.5-pro | | |
| claude-opus-4-6 | | |
| qwen/qwen3-coder-next | | |
| deepseek/deepseek-chat-v3-0324 | | |
| openai/gpt-oss-20b | | |
| openai/gpt-oss-120b | | |

### Rust

| Model | Execution Time | Speedup vs Python |
|-------|---------------|-------------------|
| gpt-5 | | |
| gemini-2.5-pro | | |
| claude-opus-4-6 | | |
| qwen/qwen3-coder-next | | |
| deepseek/deepseek-chat-v3-0324 | | |
| openai/gpt-oss-20b | | |
| openai/gpt-oss-120b | | |

### C

| Model | Execution Time | Speedup vs Python |
|-------|---------------|-------------------|
| gpt-5 | | |
| gemini-2.5-pro | | |
| claude-opus-4-6 | | |
| qwen/qwen3-coder-next | | |
| deepseek/deepseek-chat-v3-0324 | | |
| openai/gpt-oss-20b | | |
| openai/gpt-oss-120b | | |

### Go

| Model | Execution Time | Speedup vs Python |
|-------|---------------|-------------------|
| gpt-5 | | |
| gemini-2.5-pro | | |
| claude-opus-4-6 | | |
| qwen/qwen3-coder-next | | |
| deepseek/deepseek-chat-v3-0324 | | |
| openai/gpt-oss-20b | | |
| openai/gpt-oss-120b | | |

### Java

| Model | Execution Time | Speedup vs Python |
|-------|---------------|-------------------|
| gpt-5 | | |
| gemini-2.5-pro | | |
| claude-opus-4-6 | | |
| qwen/qwen3-coder-next | | |
| deepseek/deepseek-chat-v3-0324 | | |
| openai/gpt-oss-20b | | |
| openai/gpt-oss-120b | | |